# 已知产品资料，先查哪两个危险

每份公告可以有多个 RECALL-MM 危险标签。输入是已知产品大类、品名、清理后的外观及其中可见的部件词。先用分组五折检查两项提示能覆盖多少没见过的公告，再用原文把提示变成具体的检查问题。后半部分保留时间回测和其他模型对照。

外观文字来自召回图片。过滤危险措辞可以减少泄漏。结果衡量对历史公告标签的覆盖。Sentence-BERT 权重可以通过 `RECALL_MINILM_PATH` 指定。

召回原因用于理解失效、核对标签和找案例。产品开发之前能获得的大类、品名、产品描述代理（清理后的外观文字）和部件词，是这里的分类输入。

In [1]:
import json
import os
import re
import sys
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MultiLabelBinarizer, normalize
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

SEED = 2026
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_SOURCE = os.environ.get("RECALL_MINILM_PATH", MODEL_ID)
DATA_FILENAME = "dataset_corrected.json"
data_path = next((path for path in (
    Path("data") / DATA_FILENAME, Path("recall_audit/data") / DATA_FILENAME,
    Path("DM-project/recall_audit/data") / DATA_FILENAME,
) if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError("请在 recall_audit 或项目目录运行。")
sys.path.insert(0, str((data_path.parent.parent / "scripts").resolve()))
from association_rules import apriori_counts, category_hazard_rules, highlight_rules
from product_features import announcement_groups, embed_variants, product_text_variants

records = pd.DataFrame(json.loads(data_path.read_text(encoding="utf-8")))
assert records["recall_number"].is_unique
dates = pd.to_datetime(records["recall_date"])
year = dates.dt.year
train = np.flatnonzero(year.between(2000, 2013))
validate = np.flatnonzero(year.between(2014, 2018))
history = np.flatnonzero(year.between(2000, 2018))
test = np.flatnonzero(year.between(2019, 2024))
assert len(np.unique(np.r_[train, validate, test])) == len(records)
category_encoder = MultiLabelBinarizer()
hazard_encoder = MultiLabelBinarizer()
categories = category_encoder.fit_transform(records["product_classification"]).astype(bool)
labels = hazard_encoder.fit_transform(records["hazard_classification"]).astype(bool)
category_names = list(category_encoder.classes_)
hazards = list(hazard_encoder.classes_)

structure_patterns = {
    "折叠或铰链": r"foldable|folding|fold[- ]lock|fold release|hinges?|hinged",
    "轮或脚轮": r"wheels|casters?|castors?",
    "束带或安全带": r"straps?|harness(?:es)?|seat belts?",
    "外部电源线或插头": r"power cords?|power cables?|electrical plugs?|power plugs?",
    "电池仓或其盖板": r"battery compartments?|battery covers?|battery doors?",
    "拉绳或珠链": r"drawstrings?|pull cords?|bead(?:ed)? chains?",
    "门或可开启盖": r"doors?|lids?|hinged covers?",
    "通风孔或格栅": r"air vents?|ventilation slots?|grilles?|grills?",
    "外露把手": r"handles?|handgrips?|grip bars?",
}
structure_names = list(structure_patterns)
structure_re = [re.compile(rf"\b(?:{pattern})\b", re.I) for pattern in structure_patterns.values()]


fields = product_text_variants(records)
text = {column: series.map(" ".join) for column, series in fields.items()}
appearance = text["visual_product_description"]
structures = np.column_stack([
    appearance.str.contains(pattern).to_numpy() for pattern in structure_re
])
coverage = pd.DataFrame([
    {"时段": part, "公告": len(ids),
     "有可用名": int(text["product_name"].iloc[ids].ne("").sum()),
     "有可用外观": int(appearance.iloc[ids].ne("").sum()),
     "有任一结构词": int(structures[ids].any(axis=1).sum()),
     "多危险标签": int((labels[ids].sum(axis=1) > 1).sum())}
    for part, ids in [("拟合 2000–2013", train), ("验证 2014–2018", validate), ("回测 2019–2024", test)]
])
display(coverage)

,时段,公告,有可用名,有可用外观,有任一结构词,多危险标签
0,拟合 2000–2013,3791,3790,3569,1287,4
1,验证 2014–2018,1318,1317,1318,536,5
2,回测 2019–2024,1528,1528,1528,657,1


## 文本向量

品名和外观各自向量化。Sentence-BERT 先编码每个款式片段，再对一个公告的款式向量平均。词频方法从拟合公告学习 TF-IDF 词表。树模型将两个语义向量分别用 PCA 压到 32 维，与大类和结构标签组合。低维特征用来降低树模型的拟合成本。

In [2]:
semantic = embed_variants(fields, MODEL_SOURCE)
display(pd.Series({field: list(values.shape) for field, values in semantic.items()}, name="公告数、向量维数"))

product_name                  [6637, 384]
visual_product_description    [6637, 384]
Name: 公告数、向量维数, dtype: object

## 本类最常见的两个危险

A 用大类的历史危险频率。Apriori 搜索「大类＋可见结构 → 危险标签」，至少共现 5 起。置信度是该结构前提下的历史危险比例。类内提升度以同产品大类的危险频率为分母。对分母至少为 10 且存在正类内提升的前件，用本类频率收缩结构条件分布。收缩量在早期验证选择。置信度和提升度描述历史共现。因果效应的估计还需要额外设计或数据。

In [3]:
def make_features(fit_rows, query_rows):
    def design(x):
        return x[fit_rows], x[query_rows]

    category = design(categories.astype(np.float32))
    structure = design(structures.astype(np.float32))
    name, visual = [design(semantic[field]) for field in fields]

    def tfidf(column, analyzer, ngram_range):
        model = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram_range, min_df=2,
                                sublinear_tf=True, dtype=np.float32)
        return model.fit_transform(text[column].iloc[fit_rows]), model.transform(text[column].iloc[query_rows])

    name_word = tfidf("product_name", "word", (1, 2))
    name_char = tfidf("product_name", "char_wb", (3, 4))
    visual_word = tfidf("visual_product_description", "word", (1, 2))

    def combine(*pairs, dense=False):
        if dense:
            return tuple(np.column_stack(part) for part in zip(*pairs))
        return tuple(sparse.hstack(part, format="csr") for part in zip(*pairs))

    def pca(pair):
        model = PCA(n_components=32, svd_solver="randomized", random_state=SEED)
        return model.fit_transform(pair[0]), model.transform(pair[1])

    name_pca, visual_pca = pca(name), pca(visual)
    return {
        "TF-IDF 品名": combine(category, name_word, name_char),
        "TF-IDF 品名＋外观": combine(category, name_word, name_char, visual_word),
        "SBERT 品名": combine(category, name, dense=True),
        "SBERT 品名＋外观": combine(category, name, visual, dense=True),
        "SBERT 外观": combine(category, visual, dense=True),
        "低维品名": combine(category, name_pca, dense=True),
        "低维品名＋外观": combine(category, structure, name_pca, visual_pca, dense=True),
        "低维文本": combine(structure, name_pca, visual_pca, dense=True),
        "无大类 TF-IDF": combine(name_word, name_char, visual_word),
    }


def prior_scores(fit_cat, fit_y, query_cat):
    num = fit_cat.astype(np.float64).T @ fit_y
    denom = fit_cat.sum(axis=0)
    global_rate = fit_y.mean(axis=0)
    rate = np.divide(num, denom[:, None], out=np.broadcast_to(global_rate, num.shape).copy(),
                     where=denom[:, None] > 0)
    counts = query_cat.sum(axis=1, keepdims=True)
    scores = np.divide(query_cat @ rate, counts,
                       out=np.broadcast_to(global_rate, (len(query_cat), len(global_rate))).copy(),
                       where=counts > 0)
    return scores, rate


def learn_rules(fit_rows, minimum_joint=5):
    train_cat, train_struct, train_y = categories[fit_rows], structures[fit_rows], labels[fit_rows]
    cat_cols = [f"C{i}" for i in range(len(category_names))]
    struct_cols = [f"S{i}" for i in range(len(structure_names))]
    hazard_cols = [f"H{i}" for i in range(len(hazards))]
    transactions = pd.DataFrame(np.column_stack([train_cat, train_struct, train_y]),
                                columns=cat_cols + struct_cols + hazard_cols)
    counts = apriori_counts(transactions, minimum_joint, max_len=3)
    details, supported = [], {}
    for c, s, h in product(range(len(cat_cols)), range(len(struct_cols)), range(len(hazard_cols))):
        antecedent = frozenset([cat_cols[c], struct_cols[s]])
        observed = counts.get(antecedent | {hazard_cols[h]}, 0)
        if not observed:
            continue
        total = counts[antecedent]
        conf = observed / total
        base = train_y[train_cat[:, c], h].mean()
        lift_in_class = conf / base
        details.append({
            "大类": category_names[c], "结构": structure_names[s], "危险": hazards[h],
            "分母": total, "共现": observed, "置信度": conf, "类内提升度": lift_in_class,
        })
        if total >= 10 and lift_in_class > 1:
            supported[c, s] = (total, train_y[train_cat[:, c] & train_struct[:, s]].mean(axis=0))
    return pd.DataFrame(details), supported


def rule_scores(fit_rows, query_rows, supported, shrink):
    scores, base = prior_scores(categories[fit_rows], labels[fit_rows], categories[query_rows])
    combined = scores.copy()
    for q, row in enumerate(query_rows):
        per_rule = [
            (rate - base[c]) * total / (total + shrink)
            for c in np.flatnonzero(categories[row])
            for s in np.flatnonzero(structures[row])
            if (c, s) in supported
            for total, rate in [supported[c, s]]
        ]
        if per_rule:
            combined[q] += np.mean(per_rule, axis=0)
    return np.clip(combined, 0, 1)


def top_two(scores):
    winners = np.argsort(-scores, axis=1, kind="stable")[:, :2]
    result = np.zeros(scores.shape, dtype=bool)
    result[np.arange(len(scores))[:, None], winners] = True
    return result


def score_metrics(actual, predicted):
    matched = (actual & predicted).sum(axis=1)
    return {
        "命中数": int((matched > 0).sum()),
        "Hit@2": (matched > 0).mean(),
        "micro-F1": 2 * matched.sum() / (actual.sum() + predicted.sum()),
        "macro-F1": f1_score(actual, predicted, average="macro", zero_division=0),
    }

## 分类器与类别分流

对每个危险拟合一个二分类器，用分数排序应对多标签。比较伯努利朴素贝叶斯、逻辑回归、线性 SVM、近邻、Gini 决策树与 XGBoost。类别专家对样本足够的类训练同类逻辑回归，与全局模型混合。类标签已知时直接分流，隐藏类标签时用 Gini 树从品名和外观预测类别后分流。两种信息场景各自报告。软投票比较逻辑回归与 XGBoost 的概率加权。 [XGBoost 原论文](https://arxiv.org/abs/1603.02754)
    
kNN 是课内基线之一。它找文字最相近的历史公告，再让相近案例按相似度投票，直接检验「查找近邻案例」能否做出有用的危险提示。时间回测比较过 7、21、51 个近邻，最终使用 21 个，micro-F1 为 0.5191。同期 TF-IDF 逻辑回归为 0.5287。主表也在相同五折上比较这两个课内分类器。

In [4]:
def predict_estimator(estimator, x_fit, y_fit, x_query, margin=False):
    model = OneVsRestClassifier(estimator)
    model.fit(x_fit, y_fit)
    return model.decision_function(x_query) if margin else model.predict_proba(x_query)


def learner(kind, pair, y_fit, option):
    x_fit, x_query = pair
    if kind == "逻辑回归":
        return predict_estimator(LogisticRegression(C=option, max_iter=500, solver="liblinear"),
                                 x_fit, y_fit, x_query)
    if kind == "线性 SVM":
        return predict_estimator(LinearSVC(C=option, random_state=SEED),
                                 x_fit, y_fit, x_query, margin=True)
    if kind == "伯努利朴素贝叶斯":
        return predict_estimator(BernoulliNB(alpha=option), x_fit, y_fit, x_query)
    if kind == "决策树":
        return predict_estimator(DecisionTreeClassifier(max_depth=option, min_samples_leaf=20, random_state=SEED),
                                 x_fit, y_fit, x_query)
    if kind == "XGBoost":
        depth, rounds = option
        return predict_estimator(XGBClassifier(max_depth=depth, n_estimators=rounds, tree_method="hist",
                                                learning_rate=0.10, subsample=0.85, colsample_bytree=0.85,
                                                n_jobs=4, random_state=SEED, eval_metric="logloss"),
                                 x_fit, y_fit, x_query)
    if kind == "kNN":
        # Ask for a fixed number of neighbours and average their hazard labels.
        model = NearestNeighbors(n_neighbors=option, metric="cosine", algorithm="brute", n_jobs=4).fit(x_fit)
        distance, neighbors = model.kneighbors(x_query)
        weights = np.maximum(1 - distance, 0)
        weighted = (y_fit[neighbors] * weights[:, :, None]).sum(axis=1)
        denominator = weights.sum(axis=1, keepdims=True)
        return np.divide(weighted, denominator, out=y_fit[neighbors].mean(axis=1), where=denominator > 0)
    raise ValueError(kind)


def category_experts(pairs, fit_rows, query_rows):
    x_fit, x_query = pairs["无大类 TF-IDF"]
    y_fit = labels[fit_rows]
    shared = learner("逻辑回归", (x_fit, x_query), y_fit, 1.0)
    per_class = np.broadcast_to(shared, (len(category_names), *shared.shape)).copy()
    for c in range(len(category_names)):
        use = categories[fit_rows, c]
        if use.sum() < 75:
            continue
        positive = y_fit[use].sum(axis=0)
        target = (positive >= 5) & ((use.sum() - positive) >= 5)
        if target.sum() < 2:
            continue
        per_class[c][:, target] = learner("逻辑回归", (x_fit[use], x_query), y_fit[use][:, target], 0.75)

    route_known = categories[query_rows].astype(float)
    route_known /= route_known.sum(axis=1, keepdims=True)
    routed_known = np.einsum("qc,cqh->qh", route_known, per_class)

    # Predict a class from product text only when the class is withheld.
    x_fit_tree, x_query_tree = pairs["低维文本"]
    single = categories[fit_rows].sum(axis=1) == 1
    tree = DecisionTreeClassifier(max_depth=8, min_samples_leaf=15, random_state=SEED)
    tree.fit(x_fit_tree[single], categories[fit_rows][single].argmax(axis=1))
    assigned = tree.predict(x_query_tree)
    routed_tree = per_class[assigned, np.arange(len(query_rows))]
    valid_route = categories[query_rows, assigned].mean()
    return shared, routed_known, routed_tree, valid_route


def direct_confidence_predictions(fit_rows, query_rows, book, min_antecedent, min_confidence):
    """Use eligible class-association rules first and A to fill unused slots."""
    eligible = book.loc[
        book["分母"].ge(min_antecedent) & book["置信度"].ge(min_confidence),
        ["大类", "结构", "危险", "分母", "共现", "置信度"],
    ]
    routes = {}
    for category, structure, hazard, denominator, joint, confidence in eligible.itertuples(index=False, name=None):
        context = (category_names.index(category), structure_names.index(structure))
        routes.setdefault(context, []).append((hazards.index(hazard), confidence, joint, denominator, context))

    a_scores, _ = prior_scores(categories[fit_rows], labels[fit_rows], categories[query_rows])
    a_order = np.argsort(-a_scores, axis=1, kind="stable")
    prediction = np.zeros((len(query_rows), len(hazards)), dtype=bool)
    traces = []

    for row_index, source in enumerate(query_rows):
        matching = [
            rule
            for c in np.flatnonzero(categories[source])
            for s in np.flatnonzero(structures[source])
            for rule in routes.get((c, s), ())
        ]
        matching.sort(key=lambda rule: (-rule[1], -rule[2], rule[0], rule[4]))
        picked = {}
        for rule in matching:
            picked.setdefault(rule[0], rule)
            if len(picked) == 2:
                break
        traces.append(picked.copy())
        for hazard_index in a_order[row_index]:
            if len(picked) == 2:
                break
            picked.setdefault(hazard_index, None)
        prediction[row_index, list(picked)] = True
    return prediction, traces, len(eligible)

## 主评估　按产品名称分组的五折交叉验证

五折估计对同一召回资料来源中未见过的公告的覆盖。规范化后同名的公告，以及已从官方原文核实相连的公告，放在同一折。空名称用公告号单独分组。异名的其他同型号仍可能分到不同折。分层用每份公告最少见的已标危险，计分时保留所有真实标签。

主表包括课内的 kNN 与 Gini 决策树，也列出全局高频、本类频率、逻辑回归、XGBoost 和软投票。kNN 用大类和品名、外观的语义向量。Gini 树深固定为 5，和 XGBoost 使用同一套输入。另保留 Apriori 规则分类对照。每个外层训练折再划出一组内部验证公告来选择规则门槛、逻辑回归的 C、kNN 的近邻数、XGBoost 的树深和投票权重。外层测试样本只用来计分。规则、TF-IDF 词表、PCA 和分类器分别在相应训练数据上拟合。Sentence-BERT 采用固定预训练权重。每个方法始终提示两个危险。

In [5]:
from sklearn.model_selection import StratifiedGroupKFold

groups = announcement_groups(records).to_numpy()
label_counts = labels.sum(axis=0)
assert labels.any(axis=1).all()
stratum = np.where(labels, label_counts, np.inf).argmin(axis=1)
all_rows = np.arange(len(records))

rule_settings = list(product((10, 20, 30), (0.15, 0.25, 0.35)))
linear_settings = (0.5, 2.0)
boost_settings = ((3, 100), (5, 100))
neighbor_settings = (7, 21, 51)
vote_weights = (0.25, 0.5, 0.75)


def grouped_splits(rows, n_splits):
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    for fitted, withheld in splitter.split(rows, stratum[rows], groups[rows]):
        fit_rows, query_rows = rows[fitted], rows[withheld]
        assert not set(groups[fit_rows]).intersection(groups[query_rows])
        yield fit_rows, query_rows


def cv_fit_options(fit_rows, query_rows, rules, linear, boost, neighbors):
    pairs = make_features(fit_rows, query_rows)
    fit_y = labels[fit_rows]
    book, _ = learn_rules(fit_rows)
    rule_predictions = {
        setting: direct_confidence_predictions(fit_rows, query_rows, book, *setting)[0]
        for setting in rules
    }
    linear_scores = {
        c: learner("逻辑回归", pairs["TF-IDF 品名＋外观"], fit_y, c)
        for c in linear
    }
    boost_scores = {
        setting: learner("XGBoost", pairs["低维品名＋外观"], fit_y, setting)
        for setting in boost
    }
    neighbor_scores = {
        count: learner("kNN", pairs["SBERT 品名＋外观"], fit_y, count)
        for count in neighbors
    }
    return pairs, rule_predictions, linear_scores, boost_scores, neighbor_scores


def choose_by_validation(predictions, truth):
    def ranking(key):
        metrics = score_metrics(truth, predictions[key])
        return metrics["micro-F1"], metrics["macro-F1"]

    return max(predictions, key=ranking)


fold_metrics, fold_settings = [], []
oof = {}
seen = np.zeros(len(records), dtype=int)
for fold, (outer_fit, outer_test) in enumerate(grouped_splits(all_rows, 5), start=1):
    inner_fit, inner_valid = next(grouped_splits(outer_fit, 4))
    truth = labels[inner_valid]
    _, rule_candidates, lr_candidates, xgb_candidates, neighbor_candidates = cv_fit_options(
        inner_fit, inner_valid, rule_settings, linear_settings, boost_settings, neighbor_settings,
    )
    rule_choice = choose_by_validation(rule_candidates, truth)
    lr_choice = choose_by_validation({c: top_two(s) for c, s in lr_candidates.items()}, truth)
    xgb_choice = choose_by_validation({p: top_two(s) for p, s in xgb_candidates.items()}, truth)
    neighbor_choice = choose_by_validation(
        {count: top_two(scores) for count, scores in neighbor_candidates.items()}, truth
    )
    lr_valid, xgb_valid = lr_candidates[lr_choice], xgb_candidates[xgb_choice]
    vote_choice = choose_by_validation(
        {w: top_two((1-w)*lr_valid + w*xgb_valid) for w in vote_weights}, truth
    )

    pairs, rules, linear, boost, nearest = cv_fit_options(
        outer_fit, outer_test, [rule_choice], [lr_choice], [xgb_choice], [neighbor_choice],
    )
    lr_test, xgb_test = linear[lr_choice], boost[xgb_choice]
    fit_y = labels[outer_fit]
    prior, _ = prior_scores(categories[outer_fit], fit_y, categories[outer_test])
    predictions = {
        "全局高频": top_two(np.broadcast_to(fit_y.mean(axis=0), prior.shape)),
        "A 本类历史": top_two(prior),
        "Apriori 置信度": rules[rule_choice],
        "kNN": top_two(nearest[neighbor_choice]),
        "Gini 决策树": top_two(learner("决策树", pairs["低维品名＋外观"], fit_y, 5)),
        "TF-IDF 逻辑回归": top_two(lr_test),
        "XGBoost": top_two(xgb_test),
        "软投票": top_two((1-vote_choice)*lr_test + vote_choice*xgb_test),
    }
    fold_settings.append({"折": fold, "测试公告": len(outer_test), "规则门槛": str(rule_choice),
                          "kNN 近邻数": neighbor_choice, "Gini 深度": 5,
                          "逻辑回归 C": lr_choice, "XGBoost 深度": xgb_choice[0],
                          "XGBoost 投票权重": vote_choice})
    seen[outer_test] += 1
    for name, predicted in predictions.items():
        assert (predicted.sum(axis=1) == 2).all()
        oof.setdefault(name, np.zeros_like(labels))[outer_test] = predicted
        fold_metrics.append({"折": fold, "模型": name, **score_metrics(labels[outer_test], predicted)})

assert (seen == 1).all()
fold_table = pd.DataFrame(fold_metrics)
summary_cv = pd.DataFrame([
    {"模型": name, **score_metrics(labels, predicted)} for name, predicted in oof.items()
]).set_index("模型")
summary_cv["折均值 micro-F1"] = fold_table.groupby("模型")["micro-F1"].mean()
summary_cv["折间标准差"] = fold_table.groupby("模型")["micro-F1"].std()
display(summary_cv.drop(index="Apriori 置信度").round(4))
display(pd.DataFrame(fold_settings))
display(fold_table.loc[fold_table["模型"].ne("Apriori 置信度")].pivot(
    index="模型", columns="折", values="micro-F1"
).round(4))

,命中数,Hit@2,micro-F1,macro-F1,折均值 micro-F1,折间标准差
模型,,,,,,
全局高频,2687,0.4049,0.2698,0.0608,0.2698,0.0004
A 本类历史,4526,0.6819,0.4546,0.3170,0.4546,0.0037
kNN,5337,0.8041,0.5361,0.4399,0.5361,0.0065
Gini 决策树,4711,0.7098,0.4731,0.3684,0.4731,0.0109
TF-IDF 逻辑回归,5415,0.8159,0.5438,0.4470,0.5438,0.0058
XGBoost,5317,0.8011,0.5340,0.4381,0.5340,0.0021
软投票,5450,0.8212,0.5474,0.4519,0.5474,0.0026


,折,测试公告,规则门槛,kNN 近邻数,Gini 深度,逻辑回归 C,XGBoost 深度,XGBoost 投票权重
0,1,1328,"(10, 0.25)",21,5,2.0,3,0.50
1,2,1328,"(10, 0.15)",21,5,2.0,3,0.25
2,3,1327,"(10, 0.15)",21,5,2.0,5,0.25
3,4,1327,"(10, 0.15)",21,5,2.0,3,0.50
4,5,1327,"(10, 0.25)",21,5,2.0,5,0.75


折,1,2,3,4,5
模型,,,,,
A 本类历史,0.4545,0.4515,0.4505,0.4592,0.4573
Gini 决策树,0.4760,0.4660,0.4626,0.4702,0.4905
TF-IDF 逻辑回归,0.5398,0.5523,0.5374,0.5461,0.5437
XGBoost,0.5367,0.5317,0.5349,0.5320,0.5346
kNN,0.5408,0.5413,0.5274,0.5310,0.5402
全局高频,0.2704,0.2699,0.2697,0.2693,0.2696
软投票,0.5458,0.5518,0.5455,0.5461,0.5477


Hit@2 是两项提示里至少有一项和公告标签相符的比例。micro-F1 也把多报和漏报的危险标签计在内。论文和答辩可以先展示每个模型命中了多少份公告。

软投票命中 5,450/6,637 份，逻辑回归命中 5,415/6,637 份。两者的 micro-F1 分别为 0.5474 和 0.5438。

课内方法中，kNN 命中 5,337/6,637 份，micro-F1 为 0.5361。Gini 决策树命中 4,711/6,637 份，micro-F1 为 0.4731。

## 逐步加入产品信息，看看什么最有帮助

让同一个逻辑回归使用同一组五折、固定参数 C=1 和两项提示额度。输入依次为大类、大类加品名、再加外观文字、再加从外观提取的结构词。这样可以观察哪一步改善了提示。完整模型对照见上面的五折主表。

In [6]:
ablation_oof = {}
for fitted, held in grouped_splits(all_rows, 5):
    pairs = make_features(fitted, held)
    text_pair = pairs["TF-IDF 品名＋外观"]
    with_structures = tuple(
        sparse.hstack((values, structures[rows]), format="csr")
        for values, rows in zip(text_pair, (fitted, held), strict=True)
    )
    inputs = {
        "逻辑回归 · 大类": (categories[fitted], categories[held]),
        "逻辑回归 · 加品名": pairs["TF-IDF 品名"],
        "逻辑回归 · 再加外观文字": text_pair,
        "逻辑回归 · 再加结构词": with_structures,
    }
    estimates = {
        name: learner("逻辑回归", pair, labels[fitted], 1.0) for name, pair in inputs.items()
    }
    for name, scores in estimates.items():
        ablation_oof.setdefault(name, np.zeros_like(labels))[held] = top_two(scores)

input_effect = pd.DataFrame([
    {"输入版本": name, **score_metrics(labels, predictions)}
    for name, predictions in ablation_oof.items()
])
display(input_effect.round(4))

,输入版本,命中数,Hit@2,micro-F1,macro-F1
0,逻辑回归 · 大类,4526,0.6819,0.4546,0.3170
1,逻辑回归 · 加品名,5300,0.7986,0.5323,0.4350
2,逻辑回归 · 再加外观文字,5390,0.8121,0.5413,0.4429
3,逻辑回归 · 再加结构词,5393,0.8126,0.5416,0.4436


品名带来的变化最大，micro-F1 从 0.4546 升到 0.5323。再加外观文字为 0.5413，再加结构词为 0.5416，多命中 3 份公告。固定参数的输入对照和主表里的调参模型各自计分。

## 到没参与挖规则的公告里核对规律

对每一折，先用其余四折挖掘「大类 → 危险」并按发现阶段的门槛选规则，再看留出公告的**真实标签**是否仍更集中。只汇报至少在三个训练划分入选的组合。同一张表也比较软投票的两项提示，看看模型是否重现了这种模式。真实标签支持的是同数据来源中的重复表现。模型也使用了类别和历史标签，它的提示是辅助检查。两项提示的出现比例有自己的比较基准。

In [7]:
category_table = pd.DataFrame(categories, columns=category_names)
hazard_table = pd.DataFrame(labels, columns=hazards)
rule_checks = []
for fitted, held in grouped_splits(all_rows, 5):
    fitted_rules = highlight_rules(category_hazard_rules(
        category_table.iloc[fitted], hazard_table.iloc[fitted],
    ))
    for category, hazard in fitted_rules[["产品大类", "危险"]].itertuples(index=False, name=None):
        group = categories[held, category_names.index(category)]
        if not group.any():
            continue
        column = hazards.index(hazard)
        observed = labels[held, column]
        suggested = oof["软投票"][held, column]
        rule_checks.append({
            "产品大类": category, "危险": hazard, "类内公告": int(group.sum()),
            "留出公告": len(held), "类内实标": int(observed[group].sum()),
            "留出实标": int(observed.sum()), "类内提示": int(suggested[group].sum()),
            "留出提示": int(suggested.sum()),
            "实标方向相同": observed[group].mean() > observed.mean(),
            "提示方向相同": suggested[group].mean() > suggested.mean(),
        })

checks = pd.DataFrame(rule_checks)
rule_totals = []
for (category, hazard), records_for_rule in checks.groupby(["产品大类", "危险"]):
    if len(records_for_rule) < 3:
        continue
    totals = records_for_rule.sum(numeric_only=True)
    rule_totals.append({
        "产品大类": category, "危险": hazard, "入选折数": len(records_for_rule),
        "实标仍偏高的折数": int(totals["实标方向相同"]),
        "留出本类实标": f'{totals["类内实标"]}/{totals["类内公告"]}',
        "实标类内 / 整体": f'{totals["类内实标"]/totals["类内公告"]:.1%} / {totals["留出实标"]/totals["留出公告"]:.1%}',
        "提示仍偏高的折数": int(totals["提示方向相同"]),
        "提示类内 / 整体": f'{totals["类内提示"]/totals["类内公告"]:.1%} / {totals["留出提示"]/totals["留出公告"]:.1%}',
    })
rule_confirmation = pd.DataFrame(rule_totals)
display(rule_confirmation)

,产品大类,危险,入选折数,实标仍偏高的折数,留出本类实标,实标类内 / 整体,提示仍偏高的折数,提示类内 / 整体
0,BABY_PRODUCTS,CHOKING,5,5,185/402,46.0% / 14.2%,5,91.8% / 28.0%
1,CLOTHING_ACCESSORIES,CHOKING,5,5,203/425,47.8% / 14.2%,5,88.9% / 28.0%
2,ELECTRICAL,FIRE,5,5,374/594,63.0% / 24.8%,5,99.2% / 48.8%
3,FURNITURE,FALLING,5,5,271/454,59.7% / 15.6%,5,92.1% / 27.7%
4,HOME_APPLIANCES,FIRE,5,5,621/1174,52.9% / 24.8%,5,95.3% / 48.8%
5,OTHER,POISONING,5,5,132/368,35.9% / 9.4%,5,59.5% / 20.8%
6,OUTDOOR_EQUIPMENT,FIRE,5,5,172/340,50.6% / 24.8%,5,91.8% / 48.8%
7,SPORTS_RECREATION,IMPACT,5,5,148/619,23.9% / 7.3%,5,71.4% / 16.2%
8,TOOLS_HARDWARE,CUTS,5,5,123/346,35.5% / 6.7%,5,72.5% / 12.3%
9,TOYS_CHILDREN,CHOKING,5,5,493/1168,42.2% / 14.2%,5,86.6% / 28.0%


11 条代表性规则都在五个训练划分入选，五个留出折的真实标签比例也都保持「本类高于整体」。例如电气产品 374/594 份标注火灾，家用设备 621/1,174 份，整体为 24.8%。软投票的火灾提示也更集中在这两类。提示比例反映两项清单的选择。

## 从危险提示回查部件

以 Tern 折叠自行车为例，读取模型当时在留出折上给出的两项提示。再回到这个模型用过的其他产品公告中查线索。将范围缩小到名称含 bicycle 或 bicycles、且有可用外观的交通运输公告，比较外观有无提到折叠或铰链时，历史上标注跌落的比例。证据计数使用一份公告只有一条来源行的记录，便于把结构与危险对到同一条文字。公告原文另外确认实际的失效部位，作为设计检查的依据。

In [8]:
case_index = int(records.index[records["recall_number"].eq("13-231")].item())
reference, _ = next((fitted, held) for fitted, held in grouped_splits(all_rows, 5) if case_index in held)
assert groups[case_index] not in groups[reference]
bicycles = text["product_name"].str.contains(r"\bbicycles?\b", case=False, regex=True)
in_transport = categories[:, category_names.index("TRANSPORTATION")]
single_source = records["source_rows"].map(len).eq(1)
usable_appearance = appearance.ne("")
comparable = reference[(bicycles & in_transport & single_source & usable_appearance).to_numpy()[reference]]
hinge = structures[:, structure_names.index("折叠或铰链")]
fall = labels[:, hazards.index("FALLING")]
display(pd.DataFrame([{
    "留出公告": records.at[case_index, "recall_number"],
    "两项提示": ", ".join(np.array(hazards)[oof["软投票"][case_index]]),
    "公告标注": ", ".join(records.at[case_index, "hazard_classification"]),
}]))
counts = []
for mentioned, ids in (("提到折叠或铰链", comparable[hinge[comparable]]),
                       ("未提到折叠或铰链", comparable[~hinge[comparable]])):
    counts.append({"参考公告": mentioned, "标注跌落": int(fall[ids].sum()), "公告数": len(ids),
                   "占比": float(fall[ids].mean()) if len(ids) else np.nan})
design_counts = pd.DataFrame(counts)
display(design_counts.round(3))
matching = records.loc[comparable[hinge[comparable] & fall[comparable]],
                       ["recall_number", "product_name", "official"]].copy()
matching["公告里的失效位置或过程"] = matching["official"].map(lambda item: item["hazard_description"])
matching["公告原文"] = matching["official"].map(lambda item: item["url"])
with pd.option_context("display.max_colwidth", 200, "display.max_rows", 14):
    display(matching.drop(columns="official"))

,留出公告,两项提示,公告标注
0,13-231,"CRASH, FALLING",FALLING


,参考公告,标注跌落,公告数,占比
0,提到折叠或铰链,7,7,1.000
1,未提到折叠或铰链,102,151,0.675


,recall_number,product_name,公告里的失效位置或过程,公告原文
2463,09-214,2008 model year folding bicycles,"The hinge on the handlepost can crack, posing a fall hazard.",https://www.cpsc.gov/Recalls/2009/Dahon-and-REI-Novara-Folding-Bicycles-Recalled-Due-to-Fall-Hazard
3494,13-038,Folding Bicycles,"The tikit bike's handlebar stem can break and cause the rider to lose control, posing a fall hazard to the consumer.",https://www.cpsc.gov/Recalls/2013/Bike-Friday-Recalls-Tikit-Folding-Bicycles-Due-to-Fall-Hazard
3813,14-096,folding bicycles,"The bicycle's frame can break during use, posing a fall hazard to the rider.",https://www.cpsc.gov/Recalls/2014/West-Marine-Recalls-Folding-Bicycles
4336,16-068,folding bicycles,"The frame on the folding bicycles can break, posing a fall hazard.",https://www.cpsc.gov/Recalls/2016/Origin8-Recalls-Folding-Bicycles
4591,17-708,Runner bicycles with detachable pedals,"The pedals can detach from the bionic runner bicycle, posing a fall hazard to riders.&nbsp;",https://www.cpsc.gov/Recalls/2017/Running-Trainer-Recalls-Bicycles
4863,18-074,folding bicycles,"The bicycle’s bottom bracket can break during use, posing a fall hazard.",https://www.cpsc.gov/Recalls/2018/Brompton-Bicycle-Recalls-Bicycles-Due-To-Fall-Hazard
4907,18-108,Folding Bicycles,"The bike’s frame can break during use causing the bicycle to collapse, posing a fall hazard to consumers.",https://www.cpsc.gov/Recalls/2018/Allen-Sports-Recalls-Folding-Bicycles-Due-to-Fall-Hazard


这份公告的两项提示为碰撞和跌落。训练参考中，提到折叠或铰链的公告有 7/7 标注跌落，未提到的有 102/151。前一组只有 7 份，适合用来找线索。Tern 原公告指出车架上管铰链可能开裂。采用相似结构时，可以复查上管铰链在反复载荷下是否开裂。 [Tern 13-231](https://www.cpsc.gov/Recalls/2013/Stile-Products-Recalls-Tern-Folding-Bicycles)

## 补充评估　按公告时间划分

2000–2013 年拟合，2014–2018 年选配置，再用 2000–2018 年重训，在 2019–2024 年回测。这个对照回答用早期数据提示后来公告中的危险能有多准。

## 验证期选择

在共同验证集分别比较字段、向量和分类器。每个算法带入回测的配置在此阶段固定，正式选定模型也由此确定。

In [9]:
def config(kind, feature, option):
    return (kind, feature, option)


CONFIGS = (
    [config("伯努利朴素贝叶斯", "TF-IDF 品名＋外观", alpha) for alpha in (0.5, 1.0)]
    + [config(kind, feat, c) for kind in ("逻辑回归", "线性 SVM")
       for feat in ("TF-IDF 品名", "TF-IDF 品名＋外观", "SBERT 品名", "SBERT 品名＋外观", "SBERT 外观")
       for c in (0.5, 2.0)]
    + [config("决策树", "低维品名＋外观", depth) for depth in (5, 9)]
    + [config("kNN", feat, k) for feat in ("TF-IDF 品名＋外观", "SBERT 品名＋外观") for k in (7, 21, 51)]
    + [config("XGBoost", feat, setting) for feat in ("低维品名", "低维品名＋外观")
       for setting in ((3, 100), (5, 100))]
)


def evaluation_row(name, feature, parameter, scores, truth):
    return {"模型": name, "特征": feature, "参数": str(parameter), **score_metrics(truth, top_two(scores))}


def select_one(rows):
    table = pd.DataFrame(rows)
    return int(table.sort_values(["micro-F1", "macro-F1"], ascending=False, kind="stable").index[0])


def fit_and_score(fit_rows, query_rows, model_config):
    pairs = make_features(fit_rows, query_rows)
    outputs = {}
    y_fit = labels[fit_rows]
    outputs["全局高频"] = np.broadcast_to(y_fit.mean(axis=0), labels[query_rows].shape)
    outputs["A 本类先验"], _ = prior_scores(categories[fit_rows], y_fit, categories[query_rows])
    rule_book, supported = learn_rules(fit_rows)
    for shrink in (10, 30):
        outputs[("规则", shrink)] = rule_scores(fit_rows, query_rows, supported, shrink)
    for run in model_config:
        kind, feat, option = run
        outputs[run] = learner(kind, pairs[feat], y_fit, option)
    shared, known, routed, routing_hit = category_experts(pairs, fit_rows, query_rows)
    for weight in (0.5, 0.85):
        outputs[("已知类别专家", weight)] = shared * (1 - weight) + known * weight
        outputs[("Gini 预测类别后分流", weight)] = shared * (1 - weight) + routed * weight
    outputs["隐藏大类的全局模型"] = shared
    return outputs, rule_book, routing_hit


VALID = {part: labels[ids] for part, ids in [("val", validate), ("test", test)]}
val_outputs, rule_book, validation_routing = fit_and_score(train, validate, CONFIGS)
val_rows = []
val_key = []
for key, scores in val_outputs.items():
    if isinstance(key, str):
        row = evaluation_row(key, "", "", scores, VALID["val"])
    elif len(key) == 2:
        row = evaluation_row(key[0], "大类＋结构" if key[0] == "规则" else "品名＋外观与类别路由",
                             key[1], scores, VALID["val"])
    else:
        row = evaluation_row(*key, scores, VALID["val"])
    val_rows.append(row)
    val_key.append(key)

# Compare soft-voting options on validation only, using each family's chosen configuration.
by_family = {}
for family in sorted(set(row["模型"] for row in val_rows)):
    family_positions = [i for i, row in enumerate(val_rows) if row["模型"] == family]
    by_family[family] = val_key[family_positions[select_one([val_rows[i] for i in family_positions])]]

linear_key, boost_key = by_family["逻辑回归"], by_family["XGBoost"]
for weight in (0.25, 0.5, 0.75):
    key = ("软投票", weight)
    scores = (1 - weight) * val_outputs[linear_key] + weight * val_outputs[boost_key]
    val_outputs[key] = scores
    val_rows.append(evaluation_row(key[0], "验证期最好的逻辑回归＋XGBoost", weight, scores, VALID["val"]))
    val_key.append(key)
vote_positions = [i for i, key in enumerate(val_key) if isinstance(key, tuple) and key[0] == "软投票"]
by_family["软投票"] = val_key[vote_positions[select_one([val_rows[i] for i in vote_positions])]]
validation = pd.DataFrame(val_rows)
display(validation.sort_values(["micro-F1", "macro-F1"], ascending=False, kind="stable").round(4))
display(rule_book.sort_values(["共现", "类内提升度"], ascending=False).head(12).round(3))

,模型,特征,参数,命中数,Hit@2,micro-F1,macro-F1
43,软投票,验证期最好的逻辑回归＋XGBoost,0.25,1027,0.7792,0.5198,0.4135
44,软投票,验证期最好的逻辑回归＋XGBoost,0.5,1015,0.7701,0.5138,0.4109
9,逻辑回归,TF-IDF 品名＋外观,2.0,1011,0.7671,0.5117,0.4137
38,已知类别专家,品名＋外观与类别路由,0.5,1008,0.7648,0.5102,0.4070
45,软投票,验证期最好的逻辑回归＋XGBoost,0.75,1006,0.7633,0.5092,0.4082
13,逻辑回归,SBERT 品名＋外观,2.0,1005,0.7625,0.5087,0.4090
12,逻辑回归,SBERT 品名＋外观,0.5,1001,0.7595,0.5067,0.4024
40,已知类别专家,品名＋外观与类别路由,0.85,997,0.7564,0.5047,0.4068
11,逻辑回归,SBERT 品名,2.0,996,0.7557,0.5042,0.4043
32,kNN,SBERT 品名＋外观,21,996,0.7557,0.5042,0.3991


,大类,结构,危险,分母,共现,置信度,类内提升度
10,CLOTHING_ACCESSORIES,拉绳或珠链,CHOKING,95,80,0.842,1.509
30,HOME_APPLIANCES,外露把手,FIRE,136,61,0.449,0.777
25,HOME_APPLIANCES,门或可开启盖,FIRE,110,51,0.464,0.803
28,HOME_APPLIANCES,外露把手,BURN,136,43,0.316,2.159
27,HOME_APPLIANCES,通风孔或格栅,FIRE,54,43,0.796,1.379
66,TOYS_CHILDREN,外露把手,CHOKING,100,41,0.410,0.962
53,TOOLS_HARDWARE,外露把手,CUTS,95,40,0.421,1.117
58,TOYS_CHILDREN,轮或脚轮,CHOKING,85,33,0.388,0.910
44,SPORTS_RECREATION,束带或安全带,FALLING,62,30,0.484,1.199
70,TOYS_CHILDREN,外露把手,POISONING,100,30,0.300,1.032


## 后期回测

Hit@2 表示两项中至少命中一个公告标签。micro-F1 按全部标签计正确提示数和错漏。给出各方法在共同回测集上的表现，并标出验证期选定的模型。

In [10]:
# The design task assumes category membership is available. Keep the inferred route as a separate scenario.
primary = [name for name in by_family if name not in ("Gini 预测类别后分流", "隐藏大类的全局模型")]
selected = [by_family[name] for name in primary]
where_selected = [val_key.index(key) for key in selected]
winner = selected[select_one([val_rows[i] for i in where_selected])]
display(pd.DataFrame([val_rows[val_key.index(winner)]]).round(4).rename(index={0: "验证期选定"}))

# Retrain the already-chosen configurations on all pre-2019 history.
refit_configs = [key for key in selected if isinstance(key, tuple) and len(key) == 3]
test_outputs, full_rules, test_routing = fit_and_score(history, test, refit_configs)
vote_group, weight = by_family["软投票"]
test_outputs[by_family["软投票"]] = (1 - weight) * test_outputs[linear_key] + weight * test_outputs[boost_key]

chosen = [*selected, by_family["Gini 预测类别后分流"], "隐藏大类的全局模型"]
test_rows = []
for key in chosen:
    row = val_rows[val_key.index(key)].copy()
    for metric in ["命中数", "Hit@2", "micro-F1", "macro-F1"]:
        row.pop(metric)
    row["验证期选定"] = "是" if key == winner else ""
    test_rows.append({**row, **score_metrics(VALID["test"], top_two(test_outputs[key]))})
test_table = pd.DataFrame(test_rows).sort_values("micro-F1", ascending=False, kind="stable")
display(test_table[["模型", "特征", "参数", "验证期选定", "命中数", "Hit@2", "micro-F1", "macro-F1"]].round(4))
display(pd.Series({"验证期分流类别命中率": validation_routing,
                   "回测期分流类别命中率": test_routing}, name="Gini 只看文本的分类").to_frame())

,模型,特征,参数,命中数,Hit@2,micro-F1,macro-F1
验证期选定,软投票,验证期最好的逻辑回归＋XGBoost,0.25,1027,0.7792,0.5198,0.4135


,模型,特征,参数,验证期选定,命中数,Hit@2,micro-F1,macro-F1
10,软投票,验证期最好的逻辑回归＋XGBoost,0.25,是,1222,0.7997,0.5330,0.4461
9,逻辑回归,TF-IDF 品名＋外观,2.0,,1212,0.7932,0.5287,0.4358
6,已知类别专家,品名＋外观与类别路由,0.5,,1202,0.7866,0.5243,0.4312
1,XGBoost,低维品名＋外观,"(5, 100)",,1196,0.7827,0.5217,0.4424
12,隐藏大类的全局模型,,,,1194,0.7814,0.5208,0.4335
2,kNN,SBERT 品名＋外观,21,,1190,0.7788,0.5191,0.4256
3,伯努利朴素贝叶斯,TF-IDF 品名＋外观,0.5,,1181,0.7729,0.5152,0.4097
7,线性 SVM,SBERT 品名＋外观,0.5,,1172,0.7670,0.5112,0.4314
11,Gini 预测类别后分流,品名＋外观与类别路由,0.5,,1165,0.7624,0.5082,0.4152
5,决策树,低维品名＋外观,5,,1037,0.6787,0.4523,0.3530


,Gini 只看文本的分类
验证期分流类别命中率,0.631259
回测期分流类别命中率,0.622382


## 改善发生在哪些危险

输出与 A 的逐危险计数差异、两种独有命中的案例，以及同一批公告上 micro-F1 差异的配对 bootstrap 区间。区间以公告为重抽样单位，研究重复产品时可进一步按型号或召回系列划分。

In [11]:
predicted = top_two(test_outputs[winner])
baseline = top_two(test_outputs["A 本类先验"])
actual = VALID["test"]
results = pd.DataFrame({
    "危险": hazards,
    "实际数量": actual.sum(axis=0),
    "A 命中": (baseline & actual).sum(axis=0),
    "选定模型命中": (predicted & actual).sum(axis=0),
})
results["命中变化"] = results["选定模型命中"] - results["A 命中"]
display(results.sort_values("实际数量", ascending=False).reset_index(drop=True))

rng = np.random.default_rng(SEED)
draws = rng.integers(len(test), size=(2000, len(test)))
denominator = actual.sum(axis=1) + 2
actual_tp = (predicted & actual).sum(axis=1)
bootstrap_results = []
for reference, predictions in [
    ("A 本类先验", baseline), ("验证期选出的逻辑回归", top_two(test_outputs[linear_key]))
]:
    match_difference = actual_tp - (predictions & actual).sum(axis=1)
    delta = 2 * match_difference[draws].sum(axis=1) / denominator[draws].sum(axis=1)
    observed_delta = 2 * match_difference.sum() / denominator.sum()
    interval = np.quantile(delta, [0.025, 0.975])
    bootstrap_results.append({"对照": reference, "micro-F1 差": observed_delta,
                              "配对 bootstrap 下限": interval[0], "配对 bootstrap 上限": interval[1]})
interval_table = pd.DataFrame(bootstrap_results).set_index("对照")
display(interval_table.round(4))

followup = records["official"].map(lambda item: item["title"]).iloc[test].str.contains(
    r"\b(?:expand\w*|re[- ]?announc\w*)\b", regex=True, case=False
).to_numpy()
without = ~followup
if (~without).any():
    display(pd.DataFrame([
        {"子集": "标题未标扩大或重发", "公告数": int(without.sum()),
         "模型": name, **score_metrics(actual[without], matrix[without])}
        for name, matrix in [("A 本类先验", baseline), ("验证期选定", predicted)]
    ]).round(4))

def decode(matrix):
    return [", ".join(h for h, found in zip(hazards, row) if found) for row in matrix]
qualitative = records.iloc[test][["recall_number", "recall_date", "product_name"]].reset_index(drop=True)
qualitative["公告标签"] = decode(actual)
qualitative["A 提示"] = decode(baseline)
qualitative["选定模型提示"] = decode(predicted)
winner_hit, a_hit = (predicted & actual).any(axis=1), (baseline & actual).any(axis=1)
display(qualitative.loc[winner_hit & ~a_hit].head(6))
display(qualitative.loc[a_hit & ~winner_hit].head(6))

,危险,实际数量,A 命中,选定模型命中,命中变化
0,FIRE,352,275,337,62
1,FALLING,226,191,210,19
2,POISONING,201,151,168,17
3,CHOKING,143,133,137,4
4,CRASH,139,109,113,4
5,BURN,137,90,112,22
6,IMPACT,113,26,57,31
7,CUTS,86,26,47,21
8,OTHER,85,0,22,22
9,SAFETY_EQUIPMENT_MALFUNCTION,39,0,19,19


,micro-F1 差,配对 bootstrap 下限,配对 bootstrap 上限
对照,,,
A 本类先验,0.0964,0.0833,0.1095
验证期选出的逻辑回归,0.0044,-0.0013,0.0100


,子集,公告数,模型,命中数,Hit@2,micro-F1,macro-F1
0,标题未标扩大或重发,1514,A 本类先验,994,0.6565,0.4376,0.3111
1,标题未标扩大或重发,1514,验证期选定,1213,0.8012,0.5340,0.4470


,recall_number,recall_date,product_name,公告标签,A 提示,选定模型提示
11,19-070,2019-02-21,yellow diving pressure transmitters,OTHER,"FALLING, IMPACT","OTHER, SAFETY_EQUIPMENT_MALFUNCTION"
18,19-726,2019-03-05,off-highway utility vehicles,FIRE,"CRASH, FALLING","CRASH, FIRE"
22,19-080,2019-03-13,Sauté Pan with glass lid,CUTS,"BURN, FIRE","BURN, CUTS"
24,19-082,2019-03-14,Performance heated socks,FIRE,"BURN, CHOKING","BURN, FIRE"
31,19-086,2019-03-21,Wall-mounted range hoods,FALLING,"BURN, FIRE","FALLING, FIRE"
32,19-087,2019-03-21,Gun safe,SAFETY_EQUIPMENT_MALFUNCTION,"FIRE, POISONING","IMPACT, SAFETY_EQUIPMENT_MALFUNCTION"


,recall_number,recall_date,product_name,公告标签,A 提示,选定模型提示
12,19-071,2019-02-26,folding mini trampolines,IMPACT,"FALLING, IMPACT","CUTS, FALLING"
13,19-072,2019-02-26,Sleeper Chair Folding Foam Beds,FIRE,"FALLING, FIRE","CUTS, FALLING"
131,19-172,2019-07-24,10 inch table saw,FIRE,"CUTS, FIRE","CUTS, IMPACT"
143,19-180,2019-08-07,Ice Axes,FALLING,"FALLING, IMPACT","CUTS, IMPACT"
182,19-211,2019-09-26,Multi-Functional Utility Bars,CUTS,"CUTS, FIRE","FIRE, IMPACT"
233,20-712,2020-01-09,children's tricycles with canopy,POISONING,"CHOKING, POISONING","CHOKING, FALLING"


## 结论

本轮验证选定 75% 逻辑回归和 25% XGBoost 的软投票。回测 1,528 份公告中命中 1,222 份，micro-F1 为 0.5330。A 命中 1,001 份，micro-F1 为 0.4366。软投票相对 A 的回测增量为 0.0964，配对 bootstrap 区间为 [0.0833, 0.1095]。

大类、品名和外观的 TF-IDF 逻辑回归命中 1,212 份。软投票的净增为 10 份，micro-F1 差为 0.0044，区间为 [-0.0013, 0.0100]，集成是否稳定带来增量仍有不确定性。在同一批验证公告上，逻辑回归的 TF-IDF 品名配置为 0.4971，加外观的最佳配置为 0.5117。

已知类别直接路由的专家配置回测为 0.5243。隐藏类别后，Gini 树的类别命中率是 62.2%，它的级联危险提示为 0.5082。同场景的全局文本模型达到 0.5208。这为系统是否要先推断大类提供了实测依据。

按危险来看，火灾净增 62 个正确提示。热相关爆炸只有 8 个标签，本类基线和选定配置都没有命中。

回测输入的 SHA-256 为 `e90d41a29f8def54604ae6b3053cb77357c84e95b57dcd96ebeedffc1bedc37f`。

## 补充对照　如果直接用关联规则预测

这里用「产品大类＋可见结构 → 危险」作为分类关联规则。规则的 confidence 是拟合期中前件和危险同时出现的公告数，除以前件出现的公告数。新公告满足前件时才考虑该规则，先按 confidence 排序，再按共现公告数破平。取两个不同的危险标签，空位用原来的 A 本类历史频率补齐。每项由哪条规则触发都可追溯。上一轮的规则模型在命中结构后用历史样本量向 A 收缩，两种分数在同一批公告上比较。 [关联规则分类论文](https://www.aiinternational.org/Library/KDD/1998/kdd98-012.php)

沿用共现至少 5 份的 Apriori 门槛。2000–2013 年挖规则，2014–2018 年比较前件支持门槛和最小 confidence，按 micro-F1、macro-F1、较高门槛依次选定配置。用 2000–2018 年重算该配置的规则，在与前文相同的 1,528 份后期公告上做补充回测。每份公告固定提示两项。

置信度对应历史条件比例，规则箭头表示用前件推断后件。研究一个结构如何造成某个失效，需要事故机制和额外的因果证据。

In [12]:
display(summary_cv.loc[["A 本类历史", "Apriori 置信度"]].round(4))

# Select thresholds using the same validation period as the earlier experiment.
confidence_fit_book, shrink_fit = learn_rules(train)
confidence_validation = []
for min_antecedent, min_confidence in product((10, 20, 30), (0.15, 0.25, 0.35)):
    prediction, traces, n_rules = direct_confidence_predictions(
        train, validate, confidence_fit_book, min_antecedent, min_confidence
    )
    confidence_validation.append({
        "前件最少公告": min_antecedent, "最低置信度": min_confidence, "规则数": n_rules,
        "至少用一条规则": sum(bool(row) for row in traces),
        **score_metrics(labels[validate], prediction),
    })

confidence_validation = pd.DataFrame(confidence_validation).sort_values(
    ["micro-F1", "macro-F1", "前件最少公告", "最低置信度"],
    ascending=[False, False, False, False],
)
best_confidence = confidence_validation.iloc[0]
confidence_parameters = int(best_confidence["前件最少公告"]), float(best_confidence["最低置信度"])
display(confidence_validation.round(4).reset_index(drop=True))

# Reproduce the previous rule variant's selection on validation, for the side-by-side comparison.
shrink_validation = pd.DataFrame([
    {"收缩参数": shrink, **score_metrics(labels[validate], top_two(rule_scores(train, validate, shrink_fit, shrink)))}
    for shrink in (10, 30)
])
shrink_selection = int(shrink_validation.sort_values(
    ["micro-F1", "macro-F1"], ascending=False, kind="stable"
).iloc[0]["收缩参数"])

# Refit counts through 2018. Keep the early-selected thresholds for the 2019–2024 evaluation.
confidence_history_book, shrink_history = learn_rules(history)
confidence_test, confidence_trace, history_rule_count = direct_confidence_predictions(
    history, test, confidence_history_book, *confidence_parameters,
)
base_test, _ = prior_scores(categories[history], labels[history], categories[test])
confidence_comparators = {
    "A 本类先验": top_two(base_test),
    "Apriori 加本类频率收缩": top_two(rule_scores(history, test, shrink_history, shrink_selection)),
    "Apriori 直接按置信度": confidence_test,
}
confidence_baseline_table = pd.DataFrame([
    {"模型": model, **score_metrics(labels[test], matrix)}
    for model, matrix in confidence_comparators.items()
])
display(confidence_baseline_table.round(4))
display(pd.Series({
    "验证期选定的最小前件公告数": confidence_parameters[0],
    "验证期选定的最低置信度": confidence_parameters[1],
    "历史期重算后的规则数": history_rule_count,
    "回测至少一项来自规则": sum(bool(record) for record in confidence_trace),
    "回测两项都来自规则": sum(len(record) == 2 for record in confidence_trace),
}, name="直接置信度版本", dtype=object))

# Quantify the paired difference using the same bootstrap unit as the earlier report.
confidence_truth = labels[test]
denominator = confidence_truth.sum(axis=1) + 2
rng = np.random.default_rng(SEED)
bootstrap_rows = rng.integers(len(test), size=(2000, len(test)))
confidence_deltas = []
for model, other in confidence_comparators.items():
    if model == "Apriori 直接按置信度":
        continue
    difference = (confidence_test & confidence_truth).sum(axis=1) - (other & confidence_truth).sum(axis=1)
    draws = 2 * difference[bootstrap_rows].sum(axis=1) / denominator[bootstrap_rows].sum(axis=1)
    lower, upper = np.quantile(draws, [0.025, 0.975])
    confidence_deltas.append({
        "对照": model, "micro-F1 差": 2 * difference.sum() / denominator.sum(),
        "95% 区间下限": lower, "95% 区间上限": upper,
    })
confidence_intervals = pd.DataFrame(confidence_deltas)
display(confidence_intervals.round(4))

# Show which actual rules moved a hazard into the top two.
rule_examples = []
actual_hazard_names = decode(confidence_truth)
from_a = confidence_comparators["A 本类先验"]
for q, picked in enumerate(confidence_trace):
    for hazard_index, (_, confidence, joint, total, (category_index, structure_index)) in picked.items():
        if from_a[q, hazard_index]:
            continue
        rule_examples.append({
            "公告号": records.iloc[test[q]]["recall_number"],
            "产品": text["product_name"].iloc[test[q]],
            "大类": category_names[category_index], "结构": structure_names[structure_index],
            "新增危险": hazards[hazard_index], "历史共现/分母": f"{joint}/{total}",
            "历史置信度": confidence, "公告标签": actual_hazard_names[q],
            "命中": bool(confidence_truth[q, hazard_index]),
        })
confidence_examples = pd.DataFrame(rule_examples)
if not confidence_examples.empty:
    display(confidence_examples.groupby("命中", sort=False).head(4).round(3).reset_index(drop=True))

,命中数,Hit@2,micro-F1,macro-F1,折均值 micro-F1,折间标准差
模型,,,,,,
A 本类历史,4526,0.6819,0.4546,0.3170,0.4546,0.0037
Apriori 置信度,4580,0.6901,0.4600,0.3216,0.4600,0.0035


,前件最少公告,最低置信度,规则数,至少用一条规则,命中数,Hit@2,micro-F1,macro-F1
0,30,0.15,34,357,855,0.6487,0.4319,0.3162
1,20,0.15,40,390,855,0.6487,0.4319,0.3155
2,10,0.15,55,473,855,0.6487,0.4319,0.3154
3,30,0.35,18,357,851,0.6457,0.4299,0.3146
4,30,0.25,22,357,851,0.6457,0.4299,0.3146
5,20,0.35,23,390,851,0.6457,0.4299,0.3146
6,10,0.35,37,473,851,0.6457,0.4299,0.3145
7,20,0.25,28,390,851,0.6457,0.4299,0.3139
8,10,0.25,43,473,851,0.6457,0.4299,0.3138


,模型,命中数,Hit@2,micro-F1,macro-F1
0,A 本类先验,1001,0.6551,0.4366,0.3100
1,Apriori 加本类频率收缩,1025,0.6708,0.4471,0.3219
2,Apriori 直接按置信度,1026,0.6715,0.4475,0.3146


验证期选定的最小前件公告数      30
验证期选定的最低置信度      0.15
历史期重算后的规则数         43
回测至少一项来自规则        513
回测两项都来自规则         454
Name: 直接置信度版本, dtype: object

,对照,micro-F1 差,95% 区间下限,95% 区间上限
0,A 本类先验,0.0109,0.0061,0.0161
1,Apriori 加本类频率收缩,0.0004,-0.0044,0.0048


,公告号,产品,大类,结构,新增危险,历史共现/分母,历史置信度,公告标签,命中
0,19-728,off-highway utility vehicles,TRANSPORTATION,通风孔或格栅,FIRE,16/52,0.308,FIRE,True
1,19-737,walk-behind lawn mowers,OUTDOOR_EQUIPMENT,外露把手,CUTS,11/56,0.196,IMPACT,False
2,19-117,Marine coolers,OUTDOOR_EQUIPMENT,外露把手,CUTS,11/56,0.196,OTHER,False
3,19-746,recreational off-highway vehicles,TRANSPORTATION,通风孔或格栅,FIRE,16/52,0.308,CRASH,False
4,19-748,Zero turn lawn mowers,OUTDOOR_EQUIPMENT,轮或脚轮,CUTS,13/75,0.173,FIRE,False
5,20-733,Off-road vehicles with two or four seats,TRANSPORTATION,通风孔或格栅,FIRE,16/52,0.308,FIRE,True
6,20-734,Off-road vehicles with two or four seats,TRANSPORTATION,通风孔或格栅,FIRE,16/52,0.308,FIRE,True
7,20-121,baby and toddler swing,TOYS_CHILDREN,束带或安全带,FALLING,10/35,0.286,FALLING,True
